In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
print("WANDB_API_KEY length:", len(wandb_key))

WANDB_API_KEY length: 86


In [3]:
import wandb
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import transformers
import re
from pathlib import Path
from sklearn.model_selection import GroupKFold
from torch.utils.data import Dataset, DataLoader
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

torch: 2.10.0+cu128 cuda: True
transformers: 5.0.0


In [5]:
_PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Determine the correct option:",
    "Identify the correct statement:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]
_SUFFIXES = [
    "among the listed options.",
    "based on the given context.",
    "from the following choices.",
    "carefully.",
]
_PREFIX_RE = re.compile(r"^\s*(?:" + "|".join(re.escape(p) for p in _PREFIXES) + r")\s*")
_SUFFIX_RE = re.compile(r"\s*(?:" + "|".join(re.escape(s) for s in _SUFFIXES) + r")\s*$")
_WS_RE = re.compile(r"\s+")

def normalize_stem(text):
    if not isinstance(text, str):
        return text
    s = text
    while True:
        before = s
        s = _PREFIX_RE.sub("", s)
        s = _SUFFIX_RE.sub("", s)
        if s == before:
            break
    s = _WS_RE.sub(" ", s).strip()
    return s

In [6]:
comp_dir = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
train = pd.read_csv(comp_dir / "train.csv")
train["stem"] = train["prompt"].apply(normalize_stem)

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
train["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(gkf.split(train, groups=train["stem"])):
    train.loc[val_idx, "fold"] = fold_idx
print(train["fold"].value_counts().sort_index())
assert (train["fold"].value_counts() == 400).all(), "fold sizes do not match local"

fold
0    400
1    400
2    400
3    400
4    400
Name: count, dtype: int64


In [7]:
# length-debiased arm: truncate options to 60 chars to neutralize the length artifact
L_CHARS = 60
_OPT = ["A", "B", "C", "D", "E"]
for c in _OPT:
    train[c] = train[c].astype(str).str[:L_CHARS]
_ol = train[_OPT].apply(lambda col: col.str.len()).values
_yi = train["answer"].map({c: i for i, c in enumerate(_OPT)}).values
print("post-truncation correct==longest:", round((_ol.argmax(1) == _yi).mean(), 4))
assert (_ol.argmax(1) == _yi).mean() < 0.24, "length not debiased; lower L_CHARS"

post-truncation correct==longest: 0.1995


In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID = tokenizer.pad_token_id
print("vocab:", VOCAB_SIZE, "pad_id:", PAD_ID)

OPT_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {c: i for i, c in enumerate(OPT_COLS)}

class MCQPairDataset(Dataset):
    def __init__(self, df, tokenizer, stem_len=48, opt_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.stem_len = stem_len
        self.opt_len = opt_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s = self.tokenizer(row["stem"], max_length=self.stem_len, padding="max_length", truncation=True, return_tensors="pt")
        o = self.tokenizer([row[c] for c in OPT_COLS], max_length=self.opt_len, padding="max_length", truncation=True, return_tensors="pt")
        return {
            "stem_ids": s["input_ids"].squeeze(0),
            "stem_mask": s["attention_mask"].squeeze(0),
            "opt_ids": o["input_ids"],
            "opt_mask": o["attention_mask"],
            "label": LABEL2IDX[row["answer"]],
        }

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab: 50265 pad_id: 1


In [9]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNNEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, channels, kernels, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, channels, k, padding=k // 2) for k in kernels])
        self.out_dim = channels * len(kernels)

    def forward(self, ids, mask):
        x = self.embedding(ids).transpose(1, 2)
        m = mask.unsqueeze(1).bool()
        feats = []
        for conv in self.convs:
            c = F.relu(conv(x))
            c = c.masked_fill(~m, -1e9)
            feats.append(c.max(dim=2).values)
        return torch.cat(feats, dim=1)

class MCQTextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, channels, kernels, pad_id, hidden, dropout):
        super().__init__()
        self.encoder = TextCNNEncoder(vocab_size, embed_dim, channels, kernels, pad_id)
        d = self.encoder.out_dim
        self.scorer = nn.Sequential(
            nn.Linear(4 * d, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
        )

    def forward(self, stem_ids, stem_mask, opt_ids, opt_mask):
        u = self.encoder(stem_ids, stem_mask)
        b, k, l = opt_ids.shape
        v = self.encoder(opt_ids.view(b * k, l), opt_mask.view(b * k, l)).view(b, k, -1)
        u = u.unsqueeze(1).expand(-1, k, -1)
        inter = torch.cat([u, v, (u - v).abs(), u * v], dim=-1)
        return self.scorer(inter).squeeze(-1)

In [10]:
def compute_map3(logits, labels):
    top3 = logits.argsort(dim=1, descending=True)[:, :3]
    ap = torch.zeros(len(labels))
    for k in range(3):
        match = (top3[:, k] == labels)
        ap += match.float() / (k + 1)
    return ap.mean().item()

In [11]:
import random

SEED = 42
STEM_LEN = 48
OPT_LEN = 128
BATCH_SIZE = 32
EPOCHS = 15
PATIENCE = 3
LR = 1e-3
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
EMBED_DIM = 200
CHANNELS = 128
KERNELS = [3, 5, 7]
HIDDEN = 256
DROPOUT = 0.3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("config set, seed", SEED)

config set, seed 42


In [12]:
# 5-fold cv training, textcnn from scratch, hard-subset eval
import time

fold_scores = []
fold_hard = []

for fold in range(N_FOLDS):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    train_df = train[train["fold"] != fold].reset_index(drop=True)
    val_df = train[train["fold"] == fold].reset_index(drop=True)
    train_ds = MCQPairDataset(train_df, tokenizer, STEM_LEN, OPT_LEN)
    val_ds = MCQPairDataset(val_df, tokenizer, STEM_LEN, OPT_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = MCQTextCNN(VOCAB_SIZE, EMBED_DIM, CHANNELS, KERNELS, PAD_ID, HIDDEN, DROPOUT).to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    opt_lens = np.array([[len(str(val_df.iloc[r][c])) for c in OPT_COLS] for r in range(len(val_df))])
    hard = opt_lens.argmax(1) != val_df["answer"].map(LABEL2IDX).values

    run = wandb.init(
        project="23f2001849-t22026",
        entity="23f2001849-dl-genai-project",
        name=f"textcnn_fold{fold}_len60",
        group="textcnn_len60_cv",
        tags=["m2", "textcnn", "workstream-b", "scratch", "len-debiased", "cv"],
        config={
            "fold": fold, "stem_len": STEM_LEN, "opt_len": OPT_LEN, "batch_size": BATCH_SIZE,
            "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY, "embed_dim": EMBED_DIM,
            "channels": CHANNELS, "kernels": KERNELS, "hidden": HIDDEN, "dropout": DROPOUT,
            "patience": PATIENCE, "seed": SEED,
        },
    )

    best_val_map3 = -1.0
    best_hard = float("nan")
    bad_epochs = 0
    best_ckpt_path = Path(f"/kaggle/working/textcnn_fold{fold}_len60_best.pt")
    t_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        for batch in train_loader:
            stem_ids = batch["stem_ids"].to("cuda")
            stem_mask = batch["stem_mask"].to("cuda")
            opt_ids = batch["opt_ids"].to("cuda")
            opt_mask = batch["opt_mask"].to("cuda")
            labels = batch["label"].to("cuda")
            optimizer.zero_grad()
            logits = model(stem_ids, stem_mask, opt_ids, opt_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
            train_losses.append(loss.item())
            wandb.log({"train/loss": loss.item()})

        model.eval()
        val_logits_all, val_labels_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                stem_ids = batch["stem_ids"].to("cuda")
                stem_mask = batch["stem_mask"].to("cuda")
                opt_ids = batch["opt_ids"].to("cuda")
                opt_mask = batch["opt_mask"].to("cuda")
                labels = batch["label"].to("cuda")
                logits = model(stem_ids, stem_mask, opt_ids, opt_mask)
                val_logits_all.append(logits.cpu())
                val_labels_all.append(labels.cpu())
        val_logits = torch.cat(val_logits_all)
        val_labels = torch.cat(val_labels_all)
        val_map3 = compute_map3(val_logits, val_labels)
        val_map3_hard = compute_map3(val_logits[torch.from_numpy(hard)], val_labels[torch.from_numpy(hard)]) if hard.sum() else float("nan")
        train_loss = float(np.mean(train_losses))
        elapsed = time.time() - t_start
        print(f"fold {fold} epoch {epoch}: train_loss={train_loss:.4f} val_map3={val_map3:.4f} val_map3_hard={val_map3_hard:.4f} elapsed={elapsed:.1f}s")
        wandb.log({"epoch": epoch, "epoch/train_loss": train_loss, "epoch/val_map3": val_map3, "epoch/val_map3_hard": val_map3_hard})

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            best_hard = val_map3_hard
            bad_epochs = 0
            torch.save({"model_state": model.state_dict(), "val_map3": val_map3, "epoch": epoch, "fold": fold}, best_ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"  early stop at epoch {epoch}")
                break

    wandb.run.summary["best_val_map3"] = best_val_map3
    wandb.run.summary["best_val_map3_hard"] = best_hard
    run.finish()
    fold_scores.append(best_val_map3)
    fold_hard.append(best_hard)
    print(f"fold {fold} best_val_map3={best_val_map3:.4f} best_hard={best_hard:.4f}")

print("\nper-fold full:", [round(s, 4) for s in fold_scores])
print("per-fold hard:", [round(s, 4) for s in fold_hard])
print(f"cv mean full={np.mean(fold_scores):.4f}  cv mean hard={np.nanmean(fold_hard):.4f}")

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260602_033918-sw6c39h7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run textcnn_fold0_len60
wandb: ⭐️ View project at https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026
wandb: 🚀 View run at https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026/runs/sw6c39h7


fold 0 epoch 0: train_loss=0.9336 val_map3=0.4696 val_map3_hard=0.3889 elapsed=3.7s
fold 0 epoch 1: train_loss=0.5759 val_map3=0.4458 val_map3_hard=0.3853 elapsed=5.7s
fold 0 epoch 2: train_loss=0.5793 val_map3=0.4846 val_map3_hard=0.4013 elapsed=7.6s
fold 0 epoch 3: train_loss=0.5582 val_map3=0.5217 val_map3_hard=0.4258 elapsed=9.9s
fold 0 epoch 4: train_loss=0.5507 val_map3=0.4896 val_map3_hard=0.4034 elapsed=12.0s
fold 0 epoch 5: train_loss=0.5537 val_map3=0.5242 val_map3_hard=0.4611 elapsed=14.0s
fold 0 epoch 6: train_loss=0.5346 val_map3=0.5150 val_map3_hard=0.4351 elapsed=16.2s
fold 0 epoch 7: train_loss=0.5304 val_map3=0.5250 val_map3_hard=0.4600 elapsed=18.2s
fold 0 epoch 8: train_loss=0.5388 val_map3=0.5513 val_map3_hard=0.4927 elapsed=20.3s
fold 0 epoch 9: train_loss=0.5316 val_map3=0.5217 val_map3_hard=0.4330 elapsed=22.5s
fold 0 epoch 10: train_loss=0.5275 val_map3=0.5492 val_map3_hard=0.4673 elapsed=24.4s


wandb: updating run metadata


fold 0 epoch 11: train_loss=0.5197 val_map3=0.5400 val_map3_hard=0.4621 elapsed=26.3s
  early stop at epoch 11


wandb: uploading history steps 372-611, summary, console lines 7-12
wandb: 
wandb: Run history:
wandb:               epoch ▁▂▂▃▄▄▅▅▆▇▇█
wandb:    epoch/train_loss █▂▂▂▂▂▁▁▁▁▁▁
wandb:      epoch/val_map3 ▃▁▄▆▄▆▆▆█▆█▇
wandb: epoch/val_map3_hard ▁▁▂▄▂▆▄▆█▄▆▆
wandb:          train/loss █▅▅▃▄▁▂▃▃▃▅▃▂▂▃▁▃▃▂▂▃▃▂▂▃▃▃▃▁▁▂▃▂▃▃▃▂▁▃▃
wandb: 
wandb: Run summary:
wandb:       best_val_map3 0.55125
wandb:  best_val_map3_hard 0.49273
wandb:               epoch 11
wandb:    epoch/train_loss 0.51975
wandb:      epoch/val_map3 0.54
wandb: epoch/val_map3_hard 0.4621
wandb:          train/loss 0.36361
wandb: 
wandb: 🚀 View run textcnn_fold0_len60 at: https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026/runs/sw6c39h7
wandb: ⭐️ View project at: https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260602_033918-sw6c39h7/logs


...

fold 4 epoch 0: train_loss=0.9206 val_map3=0.4246 val_map3_hard=0.3862 elapsed=2.1s
fold 4 epoch 1: train_loss=0.5519 val_map3=0.4175 val_map3_hard=0.3796 elapsed=4.1s
fold 4 epoch 2: train_loss=0.5414 val_map3=0.3846 val_map3_hard=0.3481 elapsed=6.1s


wandb: updating run metadata


fold 4 epoch 3: train_loss=0.5288 val_map3=0.4150 val_map3_hard=0.3648 elapsed=8.2s
  early stop at epoch 3


wandb: uploading history steps 0-203, summary, console lines 0-4
wandb: 
wandb: Run history:
wandb:               epoch ▁▃▆█
wandb:    epoch/train_loss █▁▁▁
wandb:      epoch/val_map3 █▇▁▆
wandb: epoch/val_map3_hard █▇▁▄
wandb:          train/loss ▇▇█▇▆▃▄▂▂▃▂▃▂▂▂▂▃▃▂▂▁▂▂▃▁▄▂▁▁▂▂▂▃▃▂▂▂▁▄▂
wandb: 
wandb: Run summary:
wandb:       best_val_map3 0.42458
wandb:  best_val_map3_hard 0.38618
wandb:               epoch 3
wandb:    epoch/train_loss 0.52875
wandb:      epoch/val_map3 0.415
wandb: epoch/val_map3_hard 0.36484
wandb:          train/loss 0.46769
wandb: 
wandb: 🚀 View run textcnn_fold4_len60 at: https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026/runs/515c82fs
wandb: ⭐️ View project at: https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260602_034031-515c82fs/logs


fold 4 best_val_map3=0.4246 best_hard=0.3862

per-fold full: [0.5513, 0.4713, 0.4592, 0.5325, 0.4246]
per-fold hard: [0.4927, 0.3767, 0.4293, 0.5, 0.3862]
cv mean full=0.4878  cv mean hard=0.4370
